# Lab 1: Breast cancer classification

In this lab you will implement and train a single‑layer neural network model (affine + sigmoid) from scratch using NumPy to perform binary classification on the Breast Cancer Wisconsin dataset.

## Learning objectives
- Understand and implement the forward pass of a logistic model with the sigmoid activation.
- Derive and implement binary cross‑entropy (BCE) loss and its gradient.
- Train with mini‑batch SGD and monitor loss/accuracy.
- Evaluate on a held‑out test set and interpret results.

## What you'll do
1. Load and standardize the dataset.
2. Review `forward`, `bce_loss`, and `backward`.
3. Train with stochastic gradient descent.
4. Plot loss curves and report accuracy.
5. Visualize decision boundary


Go through the code example and complete the exercises at the bottom.



### Data loading and preprocessing
We load the Breast Cancer Wisconsin dataset with `sklearn`, split into train and test sets with stratification to preserve class balance, and standardize features using only the training statistics (mean and std). Standardization centers the features to zero mean and unit variance, which stabilizes the optimization of our logistic model.


In [ ]:
# Imports and data
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# Load
bunch = load_breast_cancer(as_frame=True)
df = bunch.frame.copy()
feature_names = bunch.feature_names.tolist()
X = df[feature_names].to_numpy(dtype=np.float64)
y = df["target"].to_numpy(dtype=np.float64).reshape(-1, 1)  # (N,1)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize using train stats only
mu = X_train.mean(axis=0, keepdims=True)
sigma = X_train.std(axis=0, keepdims=True) + 1e-8
X_train = (X_train - mu) / sigma
X_test = (X_test - mu) / sigma

N, D = X_train.shape
print(f"Train N={N}, D={D}; Test N={X_test.shape[0]}")


### Model: one-layer neural network (logistic regression)
We implement a single affine layer followed by a sigmoid activation: `y_hat = σ(XW + b)`, where `W∈R^{D×1}` and `b∈R`.
- Forward: computes logits `z = XW + b` and probabilities `y_hat = σ(z)`.
- Loss: binary cross-entropy (BCE), averaged over the batch.
- Backward: with sigmoid+BCE, the gradient w.r.t. logits is `dz = (y_hat - y) / N`, then `dW = X^T dz`, `db = sum(dz)`.
We also report accuracy by thresholding probabilities at 0.5.


#### Parameters of the network and activation (mathematical notation)

- **Weights:** $W \in \mathbb{R}^{D \times 1}$ (one parameter per input feature).
- **Bias:** $b \in \mathbb{R}$ (a scalar shared across examples).
- **Total trainable parameters:** $D + 1$.

Given standardized inputs $X \in \mathbb{R}^{N \times D}$, the model computes logits and probabilities as

$$
z = XW + b \in \mathbb{R}^{N \times 1}, \qquad \hat{y} = \sigma(z) \in (0,1)^{N \times 1}.
$$

#### Sigmoid definition and gradient

$$
\sigma(z) = \frac{1}{1 + e^{-z}} , \qquad \frac{\mathrm{d}\,\sigma}{\mathrm{d}z} = \sigma(z)\bigl(1 - \sigma(z)\bigr).
$$

With binary cross-entropy (BCE) loss and a batch of size $N$, the gradients simplify to

$$
\mathrm{d}z = \frac{\hat{y} - y}{N}, \qquad \mathrm{d}W = X^\top \mathrm{d}z, \qquad \mathrm{d}b = \mathbf{1}^\top \mathrm{d}z.
$$


In [ ]:
# One-layer NN (logistic regression) - NumPy

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Initialize parameters
rng = np.random.default_rng(42)
W = rng.normal(0.0, 0.01, size=(D, 1))
b = np.zeros((1,), dtype=np.float64)

# Forward

def forward(X):
    z = X @ W + b  # (N,1)
    y_hat = sigmoid(z)
    cache = {"X": X, "z": z, "y_hat": y_hat}
    return y_hat, cache

# Binary cross-entropy loss

def bce_loss(y_hat, y_true):
    eps = 1e-8
    y_hat = np.clip(y_hat, eps, 1 - eps)
    loss = -np.mean(y_true * np.log(y_hat) + (1 - y_true) * np.log(1 - y_hat))
    return loss

# Backward

def backward(cache, y_true):
    X = cache["X"]
    y_hat = cache["y_hat"]
    N = X.shape[0]
    dz = (y_hat - y_true) / N  # dL/dz for sigmoid+bce
    dW = X.T @ dz  # (D,1)
    db = dz.sum(axis=0)  # (1,)
    return dW, db

# Accuracy

def accuracy(y_hat, y_true, threshold=0.5):
    preds = (y_hat >= threshold).astype(np.float64)
    return float(np.mean(preds == y_true))

print("Initialized one-layer NN parameters.")


### Training: mini-batch SGD loop
At each epoch we shuffle the training data, iterate mini-batches, and perform:
1) forward pass to get `y_hat`, 2) compute BCE loss, 3) backprop to get `dW` and `db`, 4) parameter update with learning rate `lr`.
We also track train/test loss (and accuracy) per epoch to monitor convergence and generalization.

In [ ]:
# Training loop (SGD)
lr = 0.001
num_epochs = 200
batch_size = 64

N_train = X_train.shape[0]
indices = np.arange(N_train)

# Track loss per epoch
train_losses = []
test_losses = []

for epoch in range(1, num_epochs + 1):
    # Shuffle
    np.random.shuffle(indices)
    X_train = X_train[indices]
    y_train = y_train[indices]

    # Mini-batches
    for start in range(0, N_train, batch_size):
        end = start + batch_size
        Xb = X_train[start:end]
        yb = y_train[start:end]
        # Forward
        y_hat, cache = forward(Xb)
        # Loss
        loss = bce_loss(y_hat, yb)
        # Backward
        dW, db_ = backward(cache, yb)
        # Update
        W -= lr * dW
        b -= lr * db_

    # Evaluate full-batch train/test loss each epoch
    train_pred, _ = forward(X_train)
    test_pred, _ = forward(X_test)
    train_loss = bce_loss(train_pred, y_train)
    test_loss = bce_loss(test_pred, y_test)
    train_losses.append(float(train_loss))
    test_losses.append(float(test_loss))

    if epoch % 20 == 0 or epoch == 1:
        # Accuracy
        train_acc = accuracy(train_pred, y_train)
        test_acc = accuracy(test_pred, y_test)
        print(f"Epoch {epoch:03d} | loss={train_loss:.4f} | train_acc={train_acc:.3f} | test_acc={test_acc:.3f}")


In [ ]:
# Final evaluation
with np.errstate(over='ignore'):
    y_test_pred, _ = forward(X_test)

acc = accuracy(y_test_pred, y_test)
print({"test_accuracy": float(acc)})


In [ ]:
# Plot training and test loss curves
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.plot(range(1, len(train_losses)+1), train_losses, label="Train loss")
plt.plot(range(1, len(test_losses)+1), test_losses, label="Test loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.title("Loss over epochs")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Decision boundary in 2D (two features)
To visualize the learned logistic regression classifier, we can plot its decision boundary over two selected standardized features. Since the model is linear in the full 30-D space, the boundary is a hyperplane. When we restrict to two features and hold all other features fixed at their standardized mean (0), this induces a straight line in the 2D plane. We will:

- Choose two feature indices (by default the first two columns in `feature_names`).
- Create a grid over the range of those two standardized features observed in the training set.
- For each grid point, form a full 30-D standardized vector by setting the two chosen features to the grid coordinates and the remaining features to 0, then compute `p(y=1|x)` with our trained model.
- Plot the decision regions (`p>=0.5` vs `<0.5`) and overlay the training points colored by class.

Tip: You can change `i_feat` and `j_feat` to visualize other pairs of features.


In [ ]:
# 2D decision boundary visualization for two features
import numpy as np
import matplotlib.pyplot as plt

# Choose two features to visualize (indices in standardized X)
i_feat = 0  # default: first feature
j_feat = 1  # default: second feature

feat_i_name = feature_names[i_feat]
feat_j_name = feature_names[j_feat]

# Ranges from training data (already standardized)
x_i = X_train[:, i_feat]
x_j = X_train[:, j_feat]
margin = 0.5
x_min, x_max = x_i.min() - margin, x_i.max() + margin
y_min, y_max = x_j.min() - margin, x_j.max() + margin

# Meshgrid
h = 0.05
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Build full D-dim standardized inputs for grid, fixing other features at 0
num_points = xx.size
X_grid = np.zeros((num_points, D), dtype=np.float64)
X_grid[:, i_feat] = xx.ravel()
X_grid[:, j_feat] = yy.ravel()

# Predict probabilities with trained model
probs, _ = forward(X_grid)
Z = probs.reshape(xx.shape)

# Plot decision regions and training points
plt.figure(figsize=(6,5))
cs = plt.contourf(xx, yy, Z, levels=np.linspace(0, 1, 21), cmap="RdBu", alpha=0.7)
plt.colorbar(cs, label="p(y=1 | x)")

# Decision boundary p=0.5 line
plt.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=1.5)

# Overlay training points
y_train_flat = y_train.reshape(-1)
plt.scatter(x_i[y_train_flat==0], x_j[y_train_flat==0], s=18, c="#2c7fb8", edgecolor="k", linewidth=0.3, label="malignant (0)")
plt.scatter(x_i[y_train_flat==1], x_j[y_train_flat==1], s=18, c="#df65b0", edgecolor="k", linewidth=0.3, label="benign (1)")

plt.xlabel(f"{feat_i_name} (standardized)")
plt.ylabel(f"{feat_j_name} (standardized)")
plt.title("Logistic regression decision boundary")
plt.legend(frameon=True, fontsize=9)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## Exercises 

* Analyze the decision boundary above for several pairs of features
* Try to find features that seems to be easy to discriminate between the classes for
* Retrain the model with only these two features
* Report accuracy and plot the new decision boundary

